# Projeto ECAC

Trabalho realizado por:
- André Ramos, N.º 2024202276
- Simão Soares, N.º 2024222233
- Tomás Correia, N.º 2024197175

## 2. Data understanding

### 2.1 What is each column supposed to contain? Which are measurements, which are identifiers, which are administrative?

| Nome da coluna | Descrição | Tipo de dados |
|:--|:--|:--|
| Audio_Song | id do excerto de áudio, único por linha | identificador |
| Lyric_Song | id do ficheiro de letra | identificador |
| AllMusic Id	| id da faixa no catálogo AllMusic | identificador |
| Artist | artista, com uma só grafia por artista	| identificador |
| Title | título da faixa	| identificador |
| AppearancesTrackIDs	| ids de outras aparições da mesma gravação | identificador |
| AppearancesAlbumIDs	| id de álbum para cada aparição, alinhado com a anterior | identificador |
| Duration | duração da faixa completa, em segundos, > 0	| medida |
| Year, LowestYear, ActualYear | datas de lançamento, em três formatos diferentes; LowestYear ≤ ActualYear, e ActualYear não pode estar no futuro | medida |
| Moods, MoodsAll, Genres, Themes, Styles	| etiquetas do AllMusic (listas separadas por vírgulas) | medida |
| MoodsAllWeights, GenreWeights, ThemeWeights, StyleWeights | um peso por etiqueta, na mesma ordem e quantidade | medida |
| num_Genres, num_MoodsAll | contagem de etiquetas, que deve bater com as listas | medida |
| Relevance | score que o AllMusic devolveu para a query que recuperou a faixa, > 0 | medida |
| AllMusic Extraction Date | qual das duas recolhas produziu o registo (Old/New) | administrativa |
| Sample | se havia amostra de 30 s na altura da recolha | administrativa |
| SampleURL | endereço de onde o excerto foi obtido | administrativa |
| Quadrant | alvo: Q1 a Q4 | alvo |
| Arousal, Valence | alvo em forma contínua, entre 0 e 1 | alvo |

### 2.2 Do the four sources join one to one? If not, where does it break, and what follows from that?

In [ ]:
from pathlib import Path
import pandas as pd

MEDIA  = Path("dataset")
TABLES = Path("build") / "MERGE_ECAC" / "merge_ecac"

meta = pd.read_csv(TABLES / "metadata.csv")

def ids(pasta, ext):
    return {p.stem for p in (MEDIA / pasta).rglob(f"*.{ext}")}

fontes = {
    "audio": (ids("audio", "mp3"), set(meta["Audio_Song"])),
    "lyrics": (ids("lyrics", "txt"), set(meta["Lyric_Song"])),
    "covers": (ids("album_covers", "jpg"), set(meta["Audio_Song"])),
}

print("linhas:", len(meta))
print("distintas:", meta["Audio_Song"].nunique()) 
print("repetidas:", meta["Audio_Song"].duplicated().sum())

for nome, (ficheiros, tabela) in fontes.items():
    print(nome, "| ficheiros:", len(ficheiros))
    print(nome, "| tabela sem ficheiro:", len(tabela - ficheiros), "| ficheiro sem tabela:", len(ficheiros - tabela))

: 

Não. Falha em três sítios:

- Linhas repetidas na tabela. São 1348 linhas, mas só 1323 músicas distintas: 25 músicas aparecem em duas linhas.
- Áudio e letras sem linha. Todas as linhas da tabela têm áudio e letra, mas as pastas contêm 2216 ficheiros de cada tipo e a tabela só descreve 1323. Ficam 893 áudios e 893 letras sem metadados nem quadrante.
- Capas incompletas. Há 2019 capas: 109 músicas da tabela não têm capa e 805 capas não correspondem a nenhuma linha.

### 2.3 From the documentation alone, before computing anything: which columns are suspected of giving away the target? Record the list.

Moods, MoodsAll, MoodsAllWeights e num_MoodsAll são suspeitas de target leakage, pois o enunciado refere que os rótulos emocionais foram obtidos a partir de emotion tags do AllMusic. 

O caminho dos ficheiros de média (`audio/<Quadrant>/`) também revela o alvo, porque há uma pasta por quadrante. Não é coluna, mas não pode ser usado como atributo.

Themes e ThemeWeights são consideradas hipóteses a investigar, por pertencerem à mesma família de tags editoriais.

## 3. Exploratory analysis

### 3.1 Describe every column: distribution, range, missing values.

In [ ]:
from IPython.display import display

teste = pd.read_csv("build/MERGE_ECAC/merge_ecac/test_songs.csv")
metadados = pd.read_csv("build/MERGE_ECAC/merge_ecac/metadata.csv")
valores_emocionais = pd.read_csv("build/MERGE_ECAC/merge_ecac/av_values.csv")

metadados = metadados[~metadados["Audio_Song"].isin(teste["Audio_Song"])].copy()
valores_emocionais = valores_emocionais[~valores_emocionais["Audio_Song"].isin(teste["Audio_Song"])].copy()

def analisar_dados(dados, colunas_numericas):
    tabela = pd.DataFrame({
        "Tipo": dados.dtypes,
        "Em falta": dados.isna().sum(),
        "Em falta (%)": (dados.isna().mean() * 100).round(2),
        "Distintos": dados.nunique(),
        "Mais comum": dados.apply(lambda c: c.mode().iloc[0] if not c.mode().empty else None),
        "Freq. mais comum": dados.apply(lambda c: c.value_counts().max())
    })

    for coluna in colunas_numericas:
        valores = dados[coluna]

        if coluna in ["Year", "LowestYear"]:
            valores = valores.astype("string").str.extract(r"^(\d{4})", expand=False)
            
        numeros = pd.to_numeric(valores, errors="coerce")
        tabela.loc[coluna, "Mínimo"] = numeros.min()
        tabela.loc[coluna, "Máximo"] = numeros.max()
        tabela.loc[coluna, "Média"] = numeros.mean()
        tabela.loc[coluna, "Mediana"] = numeros.median()
        tabela.loc[coluna, "Desvio-padrão"] = numeros.std()

    return tabela

print("METADATA:", len(metadados), "registos,", metadados["Audio_Song"].nunique(), "músicas distintas")
display(analisar_dados(metadados, ["Relevance", "Year", "LowestYear", "Duration", "ActualYear", "num_Genres", "num_MoodsAll"]))
print("\nAV_VALUES:", len(valores_emocionais), "registos")
display(analisar_dados(valores_emocionais, ["Arousal", "Valence"]))

A análise exploratória foi realizada utilizando os ficheiros metadata.csv e av_values.csv, depois de removermos as músicas reservadas para teste. Foram analisados 932 registos de metadados, que correspondem a 914 músicas distintas, e 914 registos de valores emocionais. Para cada coluna, foram identificados o tipo de dados, os valores distintos, os valores mais frequentes e a quantidade de valores em falta. Nas colunas numéricas, calculamos também o mínimo, o máximo, a média, a mediana e o desvio-padrão.

Nos metadados, verificou-se que as colunas num_Genres e num_MoodsAll têm uma percentagem muito alta de valores em falta (98,93%), seguindo-se as colunas LowestYear (34,55%) e Duration (20,39%). Também foram encontrados valores suspeitos na coluna Duration, que varia entre -1 e 367000, e na coluna ActualYear, que varia entre 0 e 9999. Em relação à distribuição das músicas pelos quadrantes, Q1 é o mais frequente.

No ficheiro av_values.csv, não há valores em falta. A variável Arousal tem valores entre 0,0625 e 0,9750, com média 0,4940. A variável Valence tem valores entre 0,0750 e 0,9875, com média 0,5210.  Verifica-se também que Valence tem uma maior dispersão de valores. Como estas duas variáveis representam os valores emocionais que queremos prever, elas não serão usadas como características de entrada do modelo.

### 3.2 Which columns are nearly empty? Which are constant? Which measure something other than what their name says? 

In [ ]:
print("Colunas quase vazias:")
percentagem_na = metadados.isna().mean() * 100
print(percentagem_na[percentagem_na > 90].round(2))

print("\nColunas constantes:")
colunas_constantes = metadados.columns[metadados.nunique() == 1].tolist()
print(colunas_constantes)

print("\nColunas quase constantes (>90% do mesmo valor):")
for col in metadados.columns:
    maior_frequencia = metadados[col].value_counts(normalize=True).max()
    if maior_frequencia > 0.90:
        print(f"{col}: {round(maior_frequencia * 100, 2)}%")
        
display(metadados[["AllMusic Extraction Date", "Year", "LowestYear", "Sample", "Relevance"]].head(10))

A análise dos metadados permitiu observar que as colunas num_Genres e num_MoodsAll estão praticamente vazias, ambas com 98,93% de valores em falta. Não foram encontradas colunas constantes. Por outro lado, as colunas Sample e AllMusic Extraction Date apresentam pouca variabilidade, com o valor mais frequente a representar 99,42% e 92,17% dos valores disponíveis, respetivamente.

Ao comparar os dados obtidos com o ficheiro DATA_DICTIONARY.md, foi possível observar que algumas colunas apresentam nomes pouco intuitivos ou formatos diferentes dos esperados. A coluna AllMusic Extraction Date não contém datas, em vez disso, apresenta apenas os valores Old e New, que indicam de onde veio a recolha. A coluna Sample apenas indica se havia uma amostra de áudio disponível. Foram também encontradas diferenças nos formatos das datas, como é o caso da coluna Year, que deveria apresentar o formato YYYY-MM-DD, mas contém valores como 27/07/2004, e da coluna LowestYear, que deveria seguir o formato DD/MM/YYYY, mas também apresenta valores como 1994-??-??.

### 3.3 Measure how well each column alone predicts the quadrant. Some do it far too well. Which, and by what mechanism did the target reach them?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kruskal
from scipy.stats.contingency import association

train = metadados
y = train["Quadrant"]

escala = {
    col: "numerica" if pd.api.types.is_numeric_dtype(train[col]) else "nominal"
    for col in train.columns
}

def assoc_nominal(x, y):
    tab = pd.crosstab(x, y)  #Cria uma tabela de frequências entre as duas variáveis, ignorando valores NaN
    return association(tab, method="cramer", correction=True) if min(tab.shape) > 1 else np.nan  # Calcular o V de Cramér corrigido, se existirem pelo menos duas categorias em cada variável; caso contrário, devolver NaN

def assoc_numerica(x, y):
    d = pd.DataFrame({"x": x, "y": y}).dropna()  #Cria um DataFrame com as duas variáveis e remover linhas com valores omissos
    grupos = [g["x"].values for _, g in d.groupby("y")]  # Separa os valores da variável numérica de acordo com cada categoria de Quadrant

    if len(grupos) < 2 or len(d) < 2:  # Verifica se existem pelo menos dois grupos e dois registos válidos
        return np.nan  # Devolver NaN se não existirem dados suficientes para calcular a associação

    return kruskal(*grupos).statistic / (len(d) - 1)  # Aplica o teste de Kruskal-Wallis e calcula o epsilon quadrado (ε²)

# Criar uma lista com os resultados da associação de cada coluna ao Quadrant
res = pd.DataFrame(
    [(c, e, train[c].nunique(),  # Guarda o nome da coluna, a sua escala e o número de valores distintos
      (assoc_nominal if e == "nominal" else assoc_numerica)(train[c], y))  # Escolhe a função adequada e calcular a associação com o Quadrant
     for c, e in escala.items() if c != "Quadrant"],  # Percorre as colunas definidas em escala, excluindo a variável-alvo
    columns=["coluna", "escala", "n_distintos", "associacao"]  # Define os nomes das colunas da tabela de resultados
).sort_values("associacao", ascending=False)  # Ordena os resultados da maior para a menor associação

print(res.to_string(index=False))

res.dropna(subset=["associacao"]).sort_values("associacao").plot.barh(
    x="coluna",
    y="associacao",
    legend=False,
    figsize=(7, 8)
)

plt.xlabel("Associação ao Quadrant")
plt.tight_layout()
plt.show()

A análise da associação entre as variáveis e o Quadrant revelou valores muito elevados em várias colunas, nomeadamente Audio_Song, Lyric_Song, MoodsAll e Moods, que apresentaram associação perfeita. Também se observaram associações muito elevadas em variáveis como AllMusic Id, Title e Artist. Estes resultados podem indicar que algumas variáveis contêm informação diretamente relacionada com a construção do Quadrant ou identificam indiretamente os registos. 
Assim, é importante investigar a origem destas associações e verificar quais as variáveis que poderão ter contribuído para a definição da variável-alvo.

### 3.4 Compare 3.3 with the list of 2.3. What proportion was obtainable by reading alone? 

Ao comparar os resultados do exercício 3.3 com as suspeitas identificadas no 2.3, verificou-se que três das quatro colunas inicialmente consideradas suspeitas apresentaram associações elevadas com o quadrante: Moods, MoodsAll e MoodsAllWeights. Isto significa que 75% das suspeitas iniciais foram apoiadas pelos resultados obtidos, enquanto num_MoodsAll apresentou uma associação praticamente nula.

Assim, a leitura da documentação permitiu antecipar algumas das associações mais relevantes, embora outras, como as dos identificadores, só tenham sido detetadas através da análise dos dados.

### 3.5 How many songs does each artist have? How many test songs share an artist with the rest of the data?

In [ ]:
musicas_por_artista = metadados.groupby("Artist")["Audio_Song"].nunique()
musicas_por_artista = musicas_por_artista.sort_values(ascending=False)

print("Número de artistas:", len(musicas_por_artista))
print("\nTop 10 artistas com mais músicas:")
display(musicas_por_artista.head(10))

metadados_todos = pd.read_csv("build/MERGE_ECAC/merge_ecac/metadata.csv")
metadados_teste = metadados_todos[metadados_todos["Audio_Song"].isin(teste["Audio_Song"])]

artistas_treino = set(metadados["Artist"].dropna())
artistas_teste = set(metadados_teste["Artist"].dropna())
artistas_comuns = artistas_treino.intersection(artistas_teste)

musicas_comuns = metadados_teste[metadados_teste["Artist"].isin(artistas_comuns)]["Audio_Song"].nunique()

print("\nArtistas presentes no treino e no teste:", len(artistas_comuns))
print("Músicas de teste com artistas presentes no treino:", musicas_comuns)

A avaliação revelou a presença de 655 artistas distintos no conjunto de dados de treino. Destes, Billie Holiday destacou-se com o maior número de músicas (17), seguida de The Beatles e Jimmy Reed, ambos com 8 músicas.

Ao comparar os artistas do conjunto de treino com os do conjunto de teste, verificou-se que existem 104 artistas em comum. No total, 200 das 409 músicas do conjunto de teste (equivalente a 48,9%) pertencem a artistas que também estão presentes nos dados de treino.

Esta sobreposição é relevante para a avaliação do desempenho do modelo, uma vez que a presença de músicas do mesmo artista em ambos os conjuntos pode influenciar os resultados. Por esse motivo, no exercício 8, será analisada uma abordagem de validação que agrupa todas as músicas do mesmo artista, evitando esta sobreposição durante a validação cruzada.

## 4. Cleaning

### 4.1 Duplicates

#### 4.1.1 Identify the identical rows.
<!-- #endregion -->

In [ ]:
print("linhas idênticas a uma anterior:", meta.duplicated().sum())
print("linhas envolvidas:", meta.duplicated(keep=False).sum())
print("ids repetidos:", meta["Audio_Song"].duplicated().sum())

Foram identificadas 25 linhas duplicadas exatas. No total, 50 linhas estão envolvidas nestas duplicações, formando 25 pares de registos idênticos. Verificou-se ainda a existência de 25 valores repetidos de Audio_Song, correspondentes precisamente a esses pares duplicados. Como não acrescentam informação nova, estes registos podem ser removidos sem perda de informação.

#### 4.1.2 No column states which rows are the same recording. Compare four kinds of evidence: the identifier columns, the artist and title, the lyrics, the audio. Do they agree? Which of them establishes that two rows are the same recording?

In [ ]:
import hashlib

u = meta.drop_duplicates().copy()

audio_f  = {p.stem: p for p in (MEDIA / "audio").rglob("*.mp3")}
lyrics_f = {p.stem: p for p in (MEDIA / "lyrics").rglob("*.txt")}

u["art_tit"] = (u["Artist"].str.lower().str.strip() + "|" + u["Title"].str.lower().str.strip())
u["audio"] = [hashlib.md5(audio_f[i].read_bytes()).hexdigest() for i in u["Audio_Song"]]

textos = [lyrics_f[i].read_text(encoding="utf-8", errors="ignore").strip() for i in u["Lyric_Song"]]
u["letra"] = [t if len(t) > 50 else None for t in textos]

for i in ["AllMusic Id", "art_tit", "letra", "audio"]:
    print(i, "->", u[i].dropna().duplicated().sum())

print(u["AllMusic Id"].value_counts().head(1))
print(u.loc[u["AllMusic Id"] == "not_found", ["Artist", "Title"]])

for j in ["art_tit", "letra", "audio"]:
    d = u.dropna(subset=[j])
    print(f"\n{j}")
    print(d.loc[d[j].duplicated(keep=False), ["Artist", "Title", "Audio_Song", "Lyric_Song"]])

As quatro provas não concordam. Depois de remover as cópias exatas, o áudio idêntico apanha 2 pares, a letra 6 e o artista e título 1. O `AllMusic Id` não conta como prova: os seus 3 repetidos são o valor not_found em 4 músicas sem relação entre si, uma sentinela e não um identificador. 

Cada prova mede uma coisa diferente. A letra identifica a canção e não a gravação ("Twist and Shout" tem a mesma letra nos Isley Brothers e nos Beatles). O artista e título falham quando o título muda ("I F****** Hate You" e "I Fucking Hate You"). O áudio é a prova que estabelece que duas linhas são a mesma gravação. Mas só apanha cópias idênticas, e num dos 2 pares (Luther Vandross) um dos títulos diz "Remastered", pelo que o que o md5 prova é que o excerto é o mesmo.

#### 4.1.3 A remix, an edit or a live version is the same song and a different recording. Is it a duplicate?

Depende da unidade de análise. Um remix, uma edição ou uma versão ao vivo é a mesma canção, mas outra gravação: pode partilhar artista, título e letra, mas o áudio é diferente.

- Como gravação, não é duplicado. Tem características acústicas próprias, logo é distinto e não se remove.
- Como canção, as linhas não são independentes. Partilham letra e melodia e, em geral, a emoção percebida.

#### 4.1.4 Does any recording, or any song, appear on both sides of the test boundary? What does that do to the test score, and what can be done about it without changing the test partition?

In [ ]:
def atravessam(c):
    d = u.dropna(subset=[c])
    if c == "AllMusic Id":
        d = d[d[c] != "not_found"] # porque sentinela não é um identificador
    g = d.groupby(c)["teste"].agg(["min", "max"])
    return g[(g["min"] == False) & (g["max"] == True)].index

test = pd.read_csv(TABLES / "test_songs.csv")
u["teste"] = u["Audio_Song"].isin(set(test["Audio_Song"]))
print("teste:", u["teste"].sum(), "| fora do teste:", (~u["teste"]).sum()) 

for i in ["audio", "AllMusic Id", "letra", "art_tit"]:
    chaves = atravessam(i)
    n = (u[i].isin(chaves) & u["teste"]).sum()
    print(i, "| músicas de teste afetadas:", n)

for i in ["audio", "letra"]:
    print(i)
    print(u[u[i].isin(atravessam(i))][["Audio_Song", "Artist", "Title", "Duration", "Year", "teste"]].sort_values(["Artist", "Title"]))

afetadas = set()
for i in ["audio", "letra"]:
    afetadas |= set(u[u[i].isin(atravessam(i)) & u["teste"]]["Audio_Song"])
print("músicas de teste afetadas (áudio ou letra):", len(afetadas))

Sim, poucos. Uma gravação aparece dos dois lados (áudio idêntico): Godsmack, "I F****** Hate You" no treino e "I Fucking Hate You" no teste. Ao nível da canção (letra idêntica) há 3 grupos que atravessam a fronteira: o dos Godsmack, Hank Williams ("I'm So Lonesome, I Could Cry" no teste e "I'm So Lonesome" no treino) e "Look at Little Sister" (Hank Ballard no teste, Hank Ballard & the Midnighters no treino). No total, 3 das 409 músicas de teste têm um par no treino. O artista+título exato não apanha nenhum, porque o título ou o artista estão escritos de forma diferente, e o `AllMusic Id` só repete o valor not_found, que não é um identificador.

O kNN encontra um vizinho quase igual e acerta por memorização, por isso a estimativa fica otimista. Com os casos detetados, efeito é pequeno: no máximo 3/409 ≈ 0,73% das músicas de teste. Podem existir mais casos, porque o md5 e a letra exata só apanham cópias idênticas.

O que se pode fazer sem mudar o teste:

- Retirar do treino as 3 linhas que têm par no teste, ou reportar a pontuação do teste com e sem as 3 músicas afetadas (409 e 406).
- Agrupar por canção na validação cruzada.

#### 4.1.5 How many artists are written in more than one way? What does that do to a cross-validation that places all the songs of each artist in the same fold?

In [ ]:
import re, unicodedata

def limpar(nome):
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()   # tira acentos
    nome = re.sub(r"[^a-z0-9]+", " ", nome.lower()).strip()                          # minúsculas, sem pontuação nem espaços a mais
    return nome.removeprefix("the ")                                                  # tira o "the " do início

a = u.dropna(subset=["Artist"]).copy()
a["artista_norm"] = a["Artist"].map(limpar)

print("grafias distintas:", a["Artist"].nunique())
print("artistas depois de normalizar:", a["artista_norm"].nunique())

grafias = a.groupby("artista_norm")["Artist"].nunique()
print("artistas escritos de mais de uma forma:", (grafias > 1).sum())

A coluna Artist tem 835 grafias distintas, que passam a 777 artistas depois de normalizar maiúsculas, espaços, pontuação e o "The" inicial. Há 58 artistas escritos de mais de uma forma (todos com duas grafias), sempre por diferenças de maiúsculas, espaços a mais ou no fim, ou "The": por exemplo "ACE HOOD" e "Ace Hood", ou "Hank Williams" e "Hank  Williams". Há ainda pares que esta limpeza não junta, como "Cure" e "Cure, The" ou "Hank Ballard" e "Hank Ballard & the Midnighters", por isso 58 é um mínimo.

Esta validação usa Artist como grupo e coloca todas as músicas de cada grupo no mesmo fold. Se o mesmo artista tem duas grafias, o código vê dois artistas e pode pô-los em folds diferentes. Músicas do mesmo artista ficam então dos dois lados da divisão, a validação deixa de ser disjunta por artista e a estimativa fica otimista. Por isso o grupo usado é o nome normalizado.

#### 4.1.6 What should be done with each kind of duplicate? Justify

- Cópias exatas: mantém-se uma linha por par e removem-se as outras. São o mesmo registo, e deixá-las sobrepesa essas músicas. 
- Mesma gravação com ids diferentes: no par que atravessa a fronteira (Godsmack), mantém-se a linha do teste e remove-se a do treino, porque o teste não se altera e o kNN acertaria por memorização. No outro par (Luther Vandross), se estiver do mesmo lado, mantém-se uma linha, a mais completa, porque o áudio é idêntico e contaria duas vezes.
- Mesma canção, outra gravação: não se removem, porque o áudio é diferente e cada gravação tem características próprias. Ficam do mesmo lado de qualquer partição, agrupadas por canção na validação cruzada. Quando atravessam a fronteira (Hank Williams e Hank Ballard), a linha do treino sai, pelo mesmo motivo do Godsmack. 
- Total retirado do treino por fuga para o teste: 3 linhas. Nada se remove do teste.
- Artistas escritos de formas diferentes: cria-se um identificador a partir do nome normalizado, sem apagar a coluna original, e usa-se como grupo na validação por artista. 

### 4.2 Missing values

#### 4.2.1 Classify the missingness mechanism of each column. Justify from the data, not from the column name.

| Nome da coluna | Classificação de ausência | Justificação |
|:--|:--|:--|
| Duration | MNAR | Músicas mais longas têm uma maior probabilidade da sua duração ser removida. A ausência depende do valor que está ausente. |
| ActualYear | MAR | A ausência depende do número de músicas associadas com um artista, uma variável observada. |
| Genres | MAR | A ausência depende do quadrante, o variável alvo observada. A probabilidade de ausência difere entre classes. |
| Relevance | MCAR | Os valores foram removidos uniformemente e aleatoriamente entre os valores observados, com nenhuma associação detetável entre o valor original e alvo. Este é o mecanismo de controlo. |
| LowestYear | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| Year | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| Themes | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| ThemeWeights | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| AppearancesTrackIDs | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| AppearancesAlbumIDs | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| Styles | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| StyleWeights | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| Sample | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| SampleURL | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| Moods | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| MoodsAll | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| MoodsAllWeights | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| GenreWeights | Indeterminado | É requerido uma investigação mais aprofundada para estabelecer se a sua ocurrência depende nos dados observados, do valor ou de nenhum. |
| num_Genres | Não classificável | Quase inteiramente ausente, por isso não à valores observados suficientes para uma classificação empírica fiável. |
| num_MoodsAll | Não classificável | Quase inteiramente ausente, por isso não à valores observados suficientes para uma classificação empírica fiável. |

#### 4.2.2 Impute. Why should the method differ between columns?

Trocar valores inexistentes não pode ser feito de forma universal, depende do contexto de cada coluna e a razão da ausência do valor.

| Tipo de coluna | Columns | Tratamento recomendado | Justificação |
|:--|:--|:--|:--|
| Medidas numéricas | Duration, Relevance | Imputação mediana, por predefinição. | A mediana é robusta para valores extremos. No entanto, para Duration, como a ausência é do tipo MNAR, a imputação mediana pode sistematicamente subestimar a duração ausente. |
| Informação anual | ActualYear | Recuperar duma fonte de dados fiáveis e consistente sempre que possível. Caso não seja possível, usar uma mediana ou outro modelo justificado. | O ano é numérico, mas os seus valores têm de ser validados primeiro. A mediana perde informações sobre o ano de lançamento individuais. |
| Variáveis de categoria | Artist, Genres, Moods, Styles, Themes, SampleURL e a campos semelhantes, se estiverem em falta. | Usar uma categoria `Desconhecida` explícita ou um imputor adequado. | Ausência não é o mesmo que uma categoria real e a categoria mais frequente pode introduzir informações falsas. |
| Campos de múltiplos categorias | Genres, MoodsAll, Styles, Themes | Preservar a estrutura e representar que as categorias são desconhecidas quando justificável. | Uma lista não é um valor categórico singular. A imputação não pode criar categorias artificiais pesos inconsistentes. |
| Listas de pesos correspondentes | GenreWeights, MoodsAllWeights, StyleWeights, ThemeWeights | Reconciliar com a lista correspondente e validar comprimentos iguais. | Cada peso deve corresponder com o valor da mesma posição. Imputação independente pode quebrar essa relação. |
| Pares de identificadores de aparências | AppearancesTrackIDs, AppearancesAlbumIDs | Preservar a estrutura de lista de pares e usar uma representação de identificadores desconhecidos consistente. | As 2 listas descrevem informação de ligações and devem ter um número de entradas correspondentes. |
| Campos de contagem quase vazios | num_Genres, num_MoodsAll | Remover estas colunas a não ser que os seus valores previstos podem ser reconstruídos fiavelmente e a sua utilidade é justificada. | Apenas 10 valores em 1 347 são observados em cada coluna. Um imputador convencional iria fabricar valores para o dataset inteiro. |
| Identificadores e campos administrativos | Song IDs, AllMusic Id, extraction date | Não pode ser feito imputação mediana. Investigar ou excluir registos que não podem ser correspondidos fiavelmente. | Identificadores não são medidas, por isso uma estatística numérica não tem interpretação significativa. |

#### 4.2.3 One column is missing in a way that makes a particular imputation both tempting and wrong. Which, and why?

A coluna em questão é a `Duration`.

A sua ausência foi deliberadamente introduzida usando um mecanismo em que a probabilidade de remoção aumenta com a verdadeira duração. Trata-se de MNAR (Missing Not At Random). Imputação mediana é tentadora porque a duração é numérica e a mediana é robusta para valores extremos. No entanto, músicas mais longas são mais prováveis de estarem ausentes, por isso as durações observadas não são uma amostra representativa da distribuição original. Consequentemente, imputar a mediana observada para casa duração ausente vai tender a subestimar a duração real das músicas ausentes e distorcer a distribuição.

A abordagem apropriada é recuperar as durações duma fonte fiável sempre que possível. Se não for possível, usar uma imputação documentada e reconhecer a falta de precisão no resultado. Nenhum método de imputação pode recuperar inteiramente a distribuição apenas através dos valores observados sem quaisquer suposições adicionais.

### 4.3 Errors

#### 4.3.1 Identify the impossible values: sentinel codes, values outside the domain of the column, dates that have not happened.

In [ ]:
print("Duration == 0:", (meta["Duration"] == 0).sum())
print("Duration < 0:", (meta["Duration"] < 0).sum())
print("Duration == -1:", (meta["Duration"] < 0).sum())
print("Duration > 1000:", (meta["Duration"] > 1000).sum())
print("Duration == 9999:", (meta["Duration"] == 9999).sum())
print(
    "Duration between 0 and 30:",
    ((meta["Duration"] > 0) & (meta["Duration"] < 30)).sum()
)

print("Relevance <= 0: ", (meta["Relevance"] <= 0).sum())
print("Relevance is +inf: ", np.isposinf(meta["Relevance"]).sum())

years = pd.to_numeric(meta["ActualYear"], errors="coerce")
print("Malformed ActualYear:",
      (meta["ActualYear"].notna() & years.isna()).sum())
print("ActualYear == 0: ", (years == 0).sum())
print("ActualYear == 1900: ", (years == 1900).sum())
print("ActualYear == 9999: ", (years == 1900).sum())
print("ActualYear > 2026:", (years > 2026).sum())

samples = meta["Sample"].dropna()
print("Sample != 0 && Sample != 1: ", (~samples.isin([0, 1])).sum())

| Coluna | Regra | Resultados nos dados observados |
|:--|:--|:--|
| Duration | Deve ser estritamente positiva. | 110 valores são zero ou negativos: 86 zeros e 24 valores de -1. |
| Duration | Duração deve ser expressa em segundos, não milissegundos. | 73 valores excedem 1000 segundos. Inclui 60 erros de unidades, um valor anómalo e 11 valores de sentinela de 9999 (ver `truth_readme.md`). |
| Duration | A duração duma música não pode ser implausivelmente curta graças a um erro de conversão. | 26 valores positivos são abaixo de 30 segundos, muitos consistentes com durações expressas incorretamente em minutos. |
| Relevance | `Relevance` deve ser positiva e finita. | A regra foi sempre cumprida. |
| ActualYear | O ano deve ter 4 dígitos e ser válido e não pode ser do futuro. | Os dados contém 12 zeros, 10 valores de 1900, 13 valores de 9999, um malformado e 29 valores acima de 2026. |
| Sample | Deve ser binário. | Os valores observados são 0 ou 1, por isso cumprem a regra. |
| Date fields | Datas devem seguir um formato documentado e conter valores válidos. | `Year` contém múltiplos formatos. Têm de ser normalizados e rejeitar datas malformadas or impossíveis invés de tratar todas as diferenças de formato como datas impossíveis. |

#### 4.3.2 Identify the values that are impossible only next to another column, where each one alone is unremarkable. Which rule finds them?

In [ ]:
def ano(s):
    """Extrai o ano de uma data"""
    m = re.search(r"(\d{4})", str(s))  # Procura 4 dígitos seguidos
    return float(m.group(1)) if m else np.nan

lowestYears = meta["LowestYear"].map(ano)
print("ActualYear < LowestYear: ", (years < lowestYears).sum())

pairs = [
    ("Genres", "GenreWeights"),
    ("MoodsAll", "MoodsAllWeights"),
    ("Styles", "StyleWeights"),
    ("Themes", "ThemeWeights"),
]
total = 0
for tags, weights in pairs:
    both = meta[tags].notna() & meta[weights].notna()
    
    tag_count = meta[tags].str.split(",").str.len()
    weight_count = meta[weights].str.split(",").str.len()

    mismatches = both & (tag_count != weight_count)
    total += mismatches.sum()
    print(f"{tags}/{weights} mismatches: {mismatches.sum()}")

print("Total mismatches: ", total)

| Regra | Resultados observados nos dados |
|:--|:--|
| `ActualYear` não pode preceder `LowestYear`. | 55 casos de incumprimento. |
| `Genres` e `GenreWeights` devem ter o mesmo tamanho. | 12 diferenças de tamanho. |
| `MoodsAll` e `MoodsAllWeights` devem ter o mesmo tamanho. |  11 diferenças de tamanho.|
| `Styles` e `StyleWeights` devem ter o mesmo tamanho. | 11 diferenças de tamanho. |
| `Themes` e `ThemeWeights` devem ter o mesmo tamanho. | 11 diferenças de tamanho. |

### 4.4 Outliers

In [ ]:
SEED = 42
TABLES = Path("build") / "MERGE_ECAC" / "merge_ecac"

# 4.1: decisões sobre duplicados (ids identificados em 4.1.2 a 4.1.4)
# Linhas de treino que têm a mesma gravação/canção no teste. O teste não se altera, sai a do treino.
REMOVER_FRONTEIRA = ["MT0003503243",   # Godsmack, mesmo áudio que A028 (teste)
                     "MT0001134958",   # Hank Williams, mesma letra que MT0015665796 (teste)
                     "MT0005540907"]   # Hank Ballard & the Midnighters, mesma letra que MT0006540794 (teste)
LUTHER = ["MT0011933875", "A114-120"]  # mesmo áudio, os dois no treino: fica a linha mais completa

def contar_elementos(s):
    return s.fillna("").map(lambda x: len([t for t in str(x).split(",") if t.strip()]))

def remover_duplicados(df):
    """4.1.6: cópias exatas, fuga treino->teste e mesma gravação do lado do treino."""
    log = {}
    emTeste = df["Audio_Song"].isin(teste["Audio_Song"])
    exatas = df.duplicated()
    log["4.1 cópias exatas (treino)"] = int((exatas & ~emTeste).sum())
    log["4.1 cópias exatas (teste)"] = int((exatas & emTeste).sum())
    d = df[~exatas].copy()
    log["4.1 fuga treino->teste (linhas do treino)"] = int(d["Audio_Song"].isin(REMOVER_FRONTEIRA).sum())
    d = d[~d["Audio_Song"].isin(REMOVER_FRONTEIRA)]
    par = d[d["Audio_Song"].isin(LUTHER)]
    paraSair = par.isna().sum(axis=1).idxmax()  # a linha com mais células vazias
    log["4.1 mesma gravação, mesmo lado"] = 1
    return d.drop(index=paraSair), log

def corrigir_valores(df):
    """4.3: erros por regra. Devolve a tabela corrigida e quantos valores cada regra tocou."""
    d = df.copy()
    log = {}
    dur = d["Duration"]
    sentinelaDuration = dur == 9999
    nao_pos = dur <= 0
    ms = (dur >= 10000) & (dur % 1000 == 0)
    mins = (dur > 0) & (dur < 30)
    d_novo = dur.copy()
    d_novo[sentinelaDuration | nao_pos] = np.nan
    d_novo[ms] = dur[ms] / 1000
    d_novo[mins] = (dur[mins] * 60).round()
    d["Duration"] = d_novo
    log["4.3 Duration: sentinela 9999 -> NaN"] = int(sentinelaDuration.sum())
    log["4.3 Duration: zero ou negativa -> NaN"] = int(nao_pos.sum())
    log["4.3 Duration: milissegundos -> segundos"] = int(ms.sum())
    log["4.3 Duration: minutos -> segundos"] = int(mins.sum())

    ay = pd.to_numeric(d["ActualYear"], errors="coerce")
    malf = d["ActualYear"].notna() & ay.isna()
    sentinelaYear = ay.isin([0, 1900, 9999])
    futuro = (ay > 2026) & ~sentinelaYear
    ly = d["LowestYear"].map(ano)
    par_imp = (ay < ly) & ~sentinelaYear
    ay_novo = ay.copy()
    ay_novo[malf | sentinelaYear | futuro | par_imp] = np.nan
    d["ActualYear"] = ay_novo
    log["4.3 ActualYear: malformado -> NaN"] = int(malf.sum())
    log["4.3 ActualYear: sentinela 0/1900/9999 -> NaN"] = int(sentinelaYear.sum())
    log["4.3 ActualYear: no futuro -> NaN"] = int(futuro.sum())
    log["4.3 ActualYear: anterior a LowestYear -> NaN"] = int(par_imp.sum())

    for tags, pesos in [("Genres", "GenreWeights"), ("MoodsAll", "MoodsAllWeights"),
                        ("Styles", "StyleWeights"), ("Themes", "ThemeWeights")]:
        ambas = d[tags].notna() & d[pesos].notna()
        difere = ambas & (d[tags].str.split(",").str.len() != d[pesos].str.split(",").str.len())
        d.loc[difere, pesos] = np.nan
        log[f"4.3 {pesos}: tamanho diferente de {tags} -> NaN"] = int(difere.sum())
    return d, log

def acrescentar_derivadas(d):
    d = d.copy()
    d["Artist_norm"] = d["Artist"].map(lambda x: limpar(x) if pd.notna(x) else np.nan)
    d["Year_y"] = d["Year"].map(ano)
    d["LowestYear_y"] = d["LowestYear"].map(ano)
    d["n_Moods"] = contar_elementos(d["Moods"])
    d["n_MoodsAll"] = contar_elementos(d["MoodsAll"])
    d["n_Appearances"] = contar_elementos(d["AppearancesTrackIDs"])
    return d

sem_dup, log_41 = remover_duplicados(meta)
corrigido, log_43 = corrigir_valores(sem_dup)
corrigido = acrescentar_derivadas(corrigido)
e_teste = corrigido["Audio_Song"].isin(teste["Audio_Song"])
treino_c = corrigido[~e_teste].set_index("Audio_Song")
teste_c = corrigido[e_teste].set_index("Audio_Song")

print(f"treino: {len(treino_c)} músicas | teste: {len(teste_c)} músicas")
display(pd.Series({**log_41, **log_43}, name="n").to_frame())

Um erro (como foi visto em 4.3) é um valor que não pode ser verdade e encontra-se por regra. Um outlier é um valor invulgar, que pode ser perfeitamente verdadeiro. Por isso as decisões de 4.1 e as regras de 4.3 são aplicadas primeiro (funções `remover_duplicados` e `corrigir_valores` nesta célula): um `9999` deixado na coluna estragava a média e o desvio-padrão e já não deixava ver o que é invulgar a sério.

Tudo o que se ajusta aqui ajusta-se só no treino, que depois de 4.1 tem 910 músicas. O teste, com as mesmas 409 músicas, recebe os parâmetros do treino. As colunas numéricas da análise são as cinco medidas de origem (`Relevance`, `Duration`, `ActualYear`, `Year`, `LowestYear`, estas duas últimas reduzidas ao ano) e três contagens feitas a partir das listas de etiquetas (`n_Moods`, `n_MoodsAll`, `n_Appearances`), porque `num_Genres` e `num_MoodsAll` estão praticamente vazias.

#### 4.4.1 Detect outliers with univariate methods. Compare the z-score, the MAD-based modified z-score and Tukey’s fences. Which worked better, on which columns, and why do they disagree?

In [ ]:
COLS_UNI = ["Relevance", "Duration", "ActualYear", "Year_y", "LowestYear_y", "n_Moods", "n_MoodsAll", "n_Appearances"]
NOMES = {"Year_y": "Year", "LowestYear_y": "LowestYear"}

def criterios(x, ref):
    """Três critérios ajustados em ref (o treino) e aplicados a x."""
    r = ref.dropna()
    mu, sd = r.mean(), r.std()
    med = r.median()
    mad = (r - med).abs().median()
    q1, q3 = r.quantile([.25, .75])
    iqr = q3 - q1
    z = ((x - mu) / sd).abs() > 3
    modz = ((0.6745 * (x - med)) / mad).abs() > 3.5 if mad > 0 else pd.Series(False, index=x.index)
    tukey = (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)
    return z, modz, tukey, dict(mu=mu, sd=sd, med=med, mad=mad, q1=q1, q3=q3, iqr=iqr)

flags = {}  # coluna -> DataFrame de booleanos (z, modz, tukey) indexado pelas músicas de treino
parametros = {}
linhas = []
for col in COLS_UNI:
    z, modz, tukey, p = criterios(treino_c[col], treino_c[col])
    flags[col] = pd.DataFrame({"z": z, "mz": modz, "tk": tukey})
    parametros[col] = p
    buscarLog = flags[col]
    linhas.append({"coluna": NOMES.get(col, col), "n": int(treino_c[col].notna().sum()),
                   "assimetria": round(treino_c[col].skew(), 2),
                   "MAD": round(p["mad"], 4),
                   "z-score |z > 3": int(buscarLog.z.sum()), "z modificado >3.5": int(buscarLog.mz.sum()), "Tukey 1.5*IQR": int(buscarLog.tk.sum()),
                   "os três": int((buscarLog.z & buscarLog.mz & buscarLog.tk).sum()),
                   "só um": int((buscarLog.sum(axis=1) == 1).sum())})
tabela_uni = pd.DataFrame(linhas).set_index("coluna")
display(tabela_uni)

fig, eixos = plt.subplots(2, 4, figsize=(16, 7))
estilo = {"z": ("o", "tab:red", "z-score", 0.18), "mz": ("x", "tab:blue", "z modificado", 0.0), "tk": ("+", "tab:green", "Tukey", -0.18)}
for ax, col in zip(eixos.ravel(), COLS_UNI):
    x = treino_c[col].dropna()
    ax.boxplot(x, orientation="vertical", whis=1.5, showfliers=False, widths=0.4)
    for k, (m, cor, nome, dy) in estilo.items():
        v = x[flags[col][k].reindex(x.index)]
        ax.scatter(v, np.full(len(v), 1 + dy), marker=m, color=cor, s=28, label=nome, alpha=0.8)
    ax.set_title(NOMES.get(col, col)); ax.set_yticks([])
eixos[0, 0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Os critérios (|z| > 3, z modificado > 3,5 com MAD, cercas de Tukey a 1,5·IQR) discordam muito.

A discordância tem duas causas.
- O que cada critério usa como medida de dispersão. A média e o desvio-padrão são puxados pelos próprios extremos, a mediana/MAD e os quartis/IQR não. Em `Relevance` (assimetria 4,3) o desvio-padrão é 0,097 e o MAD é 0,0095, dez vezes menos. Com o desvio-padrão, o limite superior do z-score é 1,35 e só apanha o grupo à volta de 1,5 e o valor de 1,84 (33). Com o MAD e com Tukey os limites caem para 1,09 e 1,08 e passam a marcar 61 músicas entre 1,08 e 1,12, que são a cauda contínua do corpo da distribuição, sem hiato que as separe das outras.
- A assimetria. Nas colunas dos anos (assimetria perto de -1,1) o MAD e o Tukey marcam os anos antigos (todos os 34 de `Year` estão entre 1953 e 1961), que são músicas reais. Em `n_Appearances` (assimetria 5,5) a mediana é 5 e o MAD 4, e o MAD marca a partir de 27 aparições e o z-score só a partir de 96.

Qual funcionou melhor? Nenhum em todas as colunas. O z-score foi o único que não encheu de falsos alarmes as colunas de anos (0 a 1 marcas, contra 28 a 34 do MAD) e em `Duration` marca 7 durações entre 404 e 656 s, que são longas mas possíveis. Em `Relevance` e `n_Appearances` mostra a estrutura, enquanto o MAD e o Tukey marcam mais de 100 músicas que são só a cauda. Em `n_MoodsAll`, que é simétrica (assimetria 0,17), os três concordam. O Tukey tem a vantagem de apanhar também valores baixos (4 durações entre 48 e 66 s). A lição é que um ponto de corte marca o raro, não o errado. O que separa um valor isolado de uma cauda é haver ou não hiato e grupo, e isso decide-se em 4.4.3 e 4.4.5.

#### 4.4.2 Detect outliers with multivariate methods. Compare k-means and DBSCAN on a numeric feature space. Justify the parameters of each.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score

COLS_MV = ["Relevance", "n_Moods", "n_MoodsAll", "n_Appearances"]
Zmv = treino_c[COLS_MV].dropna()  # casos completos do treino
escala = StandardScaler().fit(Zmv)  # ajustada só no treino
Xmv = escala.transform(Zmv)
print(f"espaço multivariado: {len(COLS_MV)} colunas, {len(Zmv)} músicas de treino com todas as colunas ({len(Zmv)/len(treino_c):.0%})")

MIN_SAMPLES = 5
def curva_kdist(X, k):
    d = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X)[0][:, k]
    return np.sort(d)[::-1]

def joelho(y):
    """Ponto da curva (ordenada decrescente) mais afastado da corda entre os extremos."""
    x = np.arange(len(y))
    p0 = np.array([0, y[0]])
    p1 = np.array([len(y) - 1, y[-1]])
    v = (p1 - p0) / np.linalg.norm(p1 - p0)
    pts = np.c_[x, y] - p0
    dist = np.abs(pts[:, 0] * v[1] - pts[:, 1] * v[0])
    return int(dist.argmax())

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(16, 4.2))
for k in (3, MIN_SAMPLES, 11):
    a1.plot(curva_kdist(Xmv, k), label=f"k={k}")
kd = curva_kdist(Xmv, MIN_SAMPLES)
i_j = joelho(kd)
EPS = float(kd[i_j])
a1.axhline(EPS, color="k", ls="--", lw=0.8)
a1.set_title(f"DBSCAN: distância ao k-ésimo vizinho (joelho de k={MIN_SAMPLES}: eps≈{EPS:.2f})")
a1.set_xlabel("músicas ordenadas")
a1.set_ylabel("distância")
a1.legend()

ks = range(2, 11)
inercia, silh = [], []
for k in ks:
    km = KMeans(k, n_init=10, random_state=SEED).fit(Xmv)
    inercia.append(km.inertia_)
    silh.append(silhouette_score(Xmv, km.labels_))

a2.plot(list(ks), inercia, "o-")
a2.set_title("k-means: inércia (cotovelo)")
a2.set_xlabel("k")

a3.plot(list(ks), silh, "o-", color="tab:orange")
a3.set_title("k-means: silhouette")
a3.set_xlabel("k")
K_KM = list(ks)[int(np.argmax(silh))]
a3.axvline(K_KM, color="k", ls="--", lw=0.8)

plt.tight_layout()
plt.show()
print("eps =", round(EPS, 3), "| min_samples =", MIN_SAMPLES, "| k (silhouette máxima) =", K_KM, "| silhouettes:", [round(s, 3) for s in silh])

In [ ]:
db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit(Xmv)
km = KMeans(K_KM, n_init=10, random_state=SEED).fit(Xmv)

dist_centro = np.linalg.norm(Xmv - km.cluster_centers_[km.labels_], axis=1)
q1, q3 = np.percentile(dist_centro, [25, 75])
LIMITE_KM = q3 + 1.5 * (q3 - q1)  # cerca de Tukey sobre as distâncias ao centróide (treino)
mv = pd.DataFrame({"dbscan_ruido": db.labels_ == -1, "dbscan_cluster": db.labels_,
                   "km_cluster": km.labels_, "km_dist": dist_centro, "km_marca": dist_centro > LIMITE_KM}, index=Zmv.index)

print("DBSCAN: clusters", dict(zip(*np.unique(db.labels_[db.labels_ >= 0], return_counts=True))), "| ruído:", int(mv.dbscan_ruido.sum()))
print("k-means: tamanhos", dict(zip(*np.unique(km.labels_, return_counts=True))), "| limite de distância:", round(LIMITE_KM, 2), "| marcados:", int(mv.km_marca.sum()))
print("marcados pelos dois:", int((mv.dbscan_ruido & mv.km_marca).sum()), "| só DBSCAN:", int((mv.dbscan_ruido & ~mv.km_marca).sum()), "| só k-means:", int((~mv.dbscan_ruido & mv.km_marca).sum()))

ix = COLS_MV.index("Relevance"),
iy = COLS_MV.index("n_Appearances")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)
a1.scatter(Xmv[:, ix], Xmv[:, iy], c=km.labels_, cmap="tab10", s=14, alpha=0.7)
a1.scatter(Xmv[mv.km_marca.values, ix], Xmv[mv.km_marca.values, iy], marker="x", color="red", s=60, label="marcados")
a1.set_title(f"k-means (k={K_KM}), marcados: distância ao centróide > {LIMITE_KM:.2f}")
a1.legend()

cores = db.labels_
a2.scatter(Xmv[cores >= 0, ix], Xmv[cores >= 0, iy], c=cores[cores >= 0], cmap="tab10", s=14, alpha=0.7)
a2.scatter(Xmv[cores == -1, ix], Xmv[cores == -1, iy], marker="x", color="red", s=60, label="ruído")
a2.set_title(f"DBSCAN (eps={EPS:.2f}, min_samples={MIN_SAMPLES}), ruído a vermelho")
a2.legend()
for a in (a1, a2):
    a.set_xlabel("Relevance (padronizada)")
    a.set_ylabel("n_Appearances (padronizada)")
plt.tight_layout()
plt.show()

Espaço: `Relevance`, `n_Moods`, `n_MoodsAll` e `n_Appearances`, 754 músicas de treino (83 %) sem valores em falta nestas quatro colunas. As colunas de anos e `Duration` ficam de fora porque lhes falta entre 12 % e 34 %: usá-las obrigava a deitar fora 41 % do treino ou a imputar, e imputar fabrica a estrutura que se procura. As colunas são padronizadas com a média e o desvio-padrão do treino, porque os dois métodos usam distâncias e sem isso `Relevance`, com variações de 0,01, não contava.

DBSCAN: `min_samples = 5`, o menor grupo que aceitamos como população (e a regra de dimensão + 1). `eps = 0,72`, o joelho da curva da distância ao 5.º vizinho. O joelho da curva é pouco marcado, por isso em 4.4.3 testa-se a estabilidade para outros valores. Resultado: um cluster de 700 músicas, dois pequenos (9 e 5) e 40 músicas de ruído.

k-means: O k vem da silhouette, máxima em k = 2 (0,666, contra 0,424 em k = 3). O cotovelo da inércia não é nítido, por isso a silhouette decide. O k-means não tem uma ideia de outlier, e por isso marca-se a música cuja distância ao centróide ultrapassa a cerca de Tukey das distâncias do treino (2,92). Marca 44 músicas.

Os dois marcam 26 músicas em comum, o DBSCAN marca 14 que o k-means não marca e o k-means marca 18 que o DBSCAN deixa em clusters. As duas soluções estão lado a lado, nas mesmas duas colunas (`Relevance` e `n_Appearances`, padronizadas), na figura acima.

#### 4.4.3 Which of the two separates a record that is alone from a small group of records that are together and far from everything else? What does that distinction change in the decision to delete?

In [ ]:
rel = Zmv["Relevance"]
print("valores mais altos de Relevance no espaço multivariado:", sorted(rel.round(3).tolist())[-4:], "... e o maior hiato:",
      round(rel[rel < 1.4].max(), 3), "->", round(rel[(rel > 1.4) & (rel < 1.7)].min(), 3), "->", round(rel.max(), 3))
grupo = ((rel > 1.4) & (rel < 1.7)).values
sozinho = (rel > 1.7).values
print(f"grupo afastado: {grupo.sum()} músicas com Relevance entre 1.4 e 1.7 | registo sozinho: {sozinho.sum()} (Relevance {rel[sozinho].iloc[0]:.2f})\n")

def resumo(lab):
    g = pd.Series(lab[grupo]).map(lambda v: "ruído" if v == -1 else f"cluster {v}").value_counts().to_dict()
    s = ["ruído" if v == -1 else f"cluster {v}" for v in lab[sozinho]]
    return g, s

linhas = []
for ms in (3, 5, 8):
    for eps in (round(EPS, 2), 1.0, 1.5, 2.0):
        lab = DBSCAN(eps=eps, min_samples=ms).fit_predict(Xmv)
        g, s = resumo(lab)
        linhas.append({"min_samples": ms, "eps": eps, "ruído total": int((lab == -1).sum()), "grupo (32 músicas)": g, "registo sozinho": s[0]})
display(pd.DataFrame(linhas))

linhas = []
for k in (2, 3, 4, 5, 6):
    m = KMeans(k, n_init=10, random_state=SEED).fit(Xmv)
    d = np.linalg.norm(Xmv - m.cluster_centers_[m.labels_], axis=1)
    linhas.append({"k": k, "silhouette": round(silhouette_score(Xmv, m.labels_), 3),
                   "clusters do grupo": dict(pd.Series(m.labels_[grupo]).value_counts()),
                   "cluster do registo sozinho": int(m.labels_[sozinho][0]),
                   "dist. ao centróide (sozinho)": round(float(d[sozinho][0]), 2),
                   "dist. ao centróide (grupo, mediana)": round(float(np.median(d[grupo])), 2)})
display(pd.DataFrame(linhas))

g = treino_c[(treino_c["Relevance"] > 1.4) & (treino_c["Relevance"] < 1.7)]
resto = treino_c[treino_c["Relevance"] < 1.4]
print(f"\ngrupo de Relevance aproximadamente 1.5: n_Moods == n_MoodsAll em {(g.n_Moods == g.n_MoodsAll).mean():.0%} das músicas (no resto: {(resto.n_Moods == resto.n_MoodsAll).mean():.0%})")
print("recolha (AllMusic Extraction Date) do grupo:", g["AllMusic Extraction Date"].value_counts().to_dict(),
      "| no teste há", int(((teste_c["Relevance"] > 1.4) & (teste_c["Relevance"] < 1.7)).sum()), "músicas com Relevance entre 1.4 e 1.7")

Há os dois casos nos dados. Em `Relevance` a distribuição tem três níveis: o corpo (1,01 a 1,12), um grupo de 32 músicas entre 1,48 e 1,50, e uma só música em 1,84 (hiatos de 1,12 para 1,48 e de 1,50 para 1,84).

O DBSCAN separa-os. Com `min_samples` 3 ou 5 e `eps` de 1,0 em diante, o grupo forma um cluster (24 a 31 das 32 músicas) e a música de 1,84 fica sempre como ruído. O grupo só se desfaz quando o `eps` é pequeno de mais (com 0,72 e `min_samples = 5` ficam 18 em ruído) ou quando o `min_samples` (8, com `eps` 0,72) é maior do que a densidade do grupo, e aí as 32 passam a ruído. A distinção depende, portanto, desses dois parâmetros e não é automática.

O k-means não os separa. Para k de 2 a 4 o grupo e a música isolada caem no mesmo cluster, o único pequeno. A música isolada está mais longe do centróide (5,4 contra 2,3 de mediana no grupo, em k = 2), mas isso só ordena por distância, não diz se há alguém ao lado. Com o limite de 2,92 só 6 das 32 músicas do grupo são marcadas, e a isolada também.

O que muda na decisão de apagar: um registo isolado não tem nada que o corrobore, e é candidato a erro. Um grupo é uma subpopulação, e apagá-lo tira dos dados uma parte do mundo real que também vai aparecer no teste. Aqui o grupo é de facto uma subpopulação: 31 das 32 músicas têm `n_Moods == n_MoodsAll` (no resto é 30 %), todas são da recolha `Old`, e no teste há outras 24 (ver a célula acima).

#### 4.4.4 At least one record is unremarkable in every column, violates no rule, and is still isolated. Which method found it, and which did not?

In [ ]:
import itertools

COLS_PAR = COLS_UNI
# registos sem nenhuma marca univariada (nenhum dos três critérios, nenhuma coluna)
marca_uni = pd.concat([flags[c].any(axis=1).rename(c) for c in COLS_PAR], axis=1).any(axis=1)
# 2) nenhum registo da tabela corrigida viola as regras de 4.3 (verificação, não suposição)
viol = (treino_c["Duration"] <= 0) | (treino_c["Duration"] == 9999) | (treino_c["Duration"] >= 10000) \
     | (treino_c["ActualYear"] > 2026) | (treino_c["ActualYear"] < treino_c["LowestYear_y"]) | treino_c["ActualYear"].isin([0, 1900, 9999])
banais = ~marca_uni & ~viol
print(f"treino: {len(treino_c)} | com marca univariada: {int(marca_uni.sum())} | violam regras de 4.3 depois de corrigidos: {int(viol.sum())} | banais em todas as colunas: {int(banais.sum())}")

# varrimento: para cada par de colunas, distância ao 3.º vizinho (colunas padronizadas) e a sua posição no par
K_PAR = 3
melhor = {}
for a, b in itertools.combinations(COLS_PAR, 2):
    Zp = treino_c[[a, b]].dropna()
    Xp = StandardScaler().fit_transform(Zp)
    d = NearestNeighbors(n_neighbors=K_PAR + 1).fit(Xp).kneighbors(Xp)[0][:, K_PAR]
    pos = pd.Series(d, index=Zp.index).rank(pct=True)
    for i, v in pos.items():
        if v > melhor.get(i, (0, None))[0]:
            melhor[i] = (v, (a, b))
varr = pd.DataFrame({"percentil": {i: v[0] for i, v in melhor.items()}, "par": {i: " | ".join(NOMES.get(x, x) for x in v[1]) for i, v in melhor.items()}})
candidatos = varr.loc[varr.index.intersection(banais[banais].index)].sort_values("percentil", ascending=False)
display(candidatos.head(8).join(treino_c[["Artist", "Title"]]))

# os melhores candidatos, cada um no par de colunas onde se destaca
top = candidatos.head(4)
fig, eixos = plt.subplots(1, 4, figsize=(17, 4))
for ax, (i, linha) in zip(eixos, top.iterrows()):
    a, b = [{v: k for k, v in NOMES.items()}.get(x, x) for x in linha["par"].split(" | ")]
    Zp = treino_c[[a, b]].dropna()
    ax.scatter(Zp[a], Zp[b], s=8, alpha=0.35, color="grey")
    ax.scatter(treino_c.loc[i, a], treino_c.loc[i, b], s=90, color="red", marker="o", label=i)
    ax.set_xlabel(NOMES.get(a, a))
    ax.set_ylabel(NOMES.get(b, b))
    ax.set_title(f"percentil {linha['percentil']:.3f}")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

# o que cada método diz sobre cada candidato
def diz(i):
    return {"critérios univariados": "nenhum" if not marca_uni[i] else "marcado",
            "regras de 4.3": "nenhuma" if not viol[i] else "viola",
            "k-means (4 col.)": ("marca" if mv.loc[i, "km_marca"] else "não marca") if i in mv.index else "—",
            "DBSCAN (4 col.)": ("ruído" if mv.loc[i, "dbscan_ruido"] else "cluster") if i in mv.index else "—",
            "varrimento por pares": f"percentil {candidatos.loc[i, 'percentil']:.3f} em {candidatos.loc[i, 'par']}"}
display(pd.DataFrame({i: diz(i) for i in top.index}).T.join(treino_c[["Artist", "Title"]]))

Dos 910 registos de treino, 240 têm alguma marca univariada e nenhum viola as regras de 4.3 depois de corrigido. Ficam 670 registos banais em todas as colunas. Para cada par das 8 colunas (28 pares) mede-se a distância ao 3.º vizinho com as colunas padronizadas e vê-se a posição do registo nessa lista. Os quatro mais isolados são, nesse varrimento, `MT0029861029` (percentil 99,8 em `Year`,`LowestYear`), `MT0003536193`, `A171` e **`MT0033459750` (Donovan, Epistle to Derroll, percentil 99,6 em `Duration`,`LowestYear`).

O registo de `MT0033459750` tem `Duration` 345 s, `Year` 1967, `LowestYear` 1962 e `ActualYear` 1967. Nenhum dos valores sai das cercas e nenhuma regra falha. Quem o encontrou foi o varrimento por pares, porque é o único método que olha para a combinação de `Duration` e `LowestYear`. Não o encontraram:

- os três critérios univariados e as regras de 4.3 (não há nada de invulgar numa coluna);
- o **k-means** de 4.4.2 (distância ao centróide de 1,77, abaixo do limite de 2,92);
- o **DBSCAN** de 4.4.2 (fica dentro do cluster grande).

Estes dois últimos não o encontram porque o espaço de 4.4.2 não tem as colunas dos anos nem `Duration`. A escolha do espaço decide o que se vê.

O varrimento propõe hipóteses, não prova erros. Comparando os quatro candidatos com `merge_original`, o primeiro é uma reedição verdadeira (`Year` 2008, `LowestYear` 1978, sem nenhuma alteração), o segundo e o terceiro só diferem pelos defeitos que as regras de 4.3 e a falta de valores já tratam, e só o quarto tem um valor alterado que nenhuma regra apanha: `LowestYear` publicado como 1967-12-??.

#### 4.4.5 Which of the flagged records are extreme and correct, and must survive the filter? How does the criterion keep them?

In [ ]:
COLS_MEDIDA = ["Relevance", "Duration", "ActualYear", "Year_y", "LowestYear_y"]
# 6*MAD é aproximadamente 4 desvios-padrão
G = 6
MIN_GRUPO = 5

def ajustar_caudas(serie):
    """Aprende, só no treino, o intervalo interior da coluna (sem as caudas isoladas)."""
    v = np.sort(serie.dropna().values)
    mad = np.median(np.abs(v - np.median(v)))
    lo = v[0]
    hi = v[-1]
    for t in range(1, MIN_GRUPO):  # cauda superior de t valores
        if v[-t] - v[-t - 1] > G * mad:
            hi = v[-t - 1]
    for t in range(1, MIN_GRUPO):  # cauda inferior de t valores
        if v[t] - v[t - 1] > G * mad:
            lo = v[t]
    return dict(lo=lo, hi=hi, folga=G * mad)

def isolados(serie, p):
    return (serie > p["hi"] + p["folga"]) | (serie < p["lo"] - p["folga"])

caudas = {c: ajustar_caudas(treino_c[c]) for c in COLS_MEDIDA}
linhas = []
for col in COLS_MEDIDA:
    p = caudas[col]
    ft = isolados(treino_c[col], p)
    fe = isolados(teste_c[col], p)
    linhas.append({"coluna": NOMES.get(col, col), "intervalo interior": f"[{p['lo']:.4g}, {p['hi']:.4g}]", "folga (6·MAD)": round(p["folga"], 4), "isolados no treino": int(ft.sum()), "isolados no teste": int(fe.sum())})
display(pd.DataFrame(linhas).set_index("coluna"))

for col in COLS_MEDIDA:
    for nome, d in (("treino", treino_c), ("teste", teste_c)):
        m = isolados(d[col], caudas[col])
        if m.any():
            print(f"{NOMES.get(col, col)} | {nome}:", d.loc[m, col].to_dict())

# os extremos que o critério deixa passar: marcados pelos três critérios univariados e não isolados
tres = {c: flags[c].all(axis=1) for c in COLS_MEDIDA}
sobrevivem = pd.DataFrame({NOMES.get(c, c): [int(tres[c].sum()), int((tres[c] & ~isolados(treino_c[c], caudas[c])).sum())] for c in COLS_MEDIDA}, index=["marcados pelos 3 critérios", "...e que sobrevivem ao critério"])
display(sobrevivem)

Extremos e corretos, que têm de sobreviver:
- as 32 músicas de `Relevance` entre 1,48 e 1,50, marcadas pelos três critérios, mas todas iguais à tabela publicada e com a estrutura de subpopulação de 4.4.3;
- as durações longas marcadas pelo z-score (404 a 656 s), por exemplo `MT0001435061`, com 656 s, igual à publicada;
- as músicas com muitas aparições (`n_Appearances` até 297), que são contagens verdadeiras de gravações muito populares;
- os anos antigos (1953 a 1961 em `Year`) que o MAD e o Tukey marcam.

Critério: valor isolado e sem grupo. Numa coluna medida na origem (`Relevance`, `Duration`, `ActualYear`, `Year`, `LowestYear`), um valor é isolado se pertence a uma cauda de menos de 5 valores separada do resto por um hiato maior do que 6·MAD (cerca de 4 desvios-padrão, porque sigma é aproximadamente 1,4826·MAD). Uma cauda com 5 ou mais valores é um grupo e sobrevive. As contagens tiradas das listas de etiquetas nunca se tocam, porque os seus extremos são reais. O intervalo interior aprende-se só no treino e aplica-se ao teste. O DBSCAN e o k-means não apagam nada: ficam como marcas informativas (`isolado_dbscan`, `longe_kmeans`, `n_criterios_uni`), porque num espaço sem os anos nem `Duration` marcam o grupo de 1,5 e são um mau juiz sozinhos.

Resultado: no treino só o `Relevance` = 1,84 (`MT0004184616`) é isolado (hiato de 0,34 contra uma folga de 0,057). No teste só uma `Duration` de 1680 s (a maior do treino é 656 s). Das 33 músicas de `Relevance` marcadas pelos três critérios, 32 sobrevivem, e os 7 extremos de `Duration` também. O critério não vê erros dentro do corpo da distribuição (isso é o 4.4.4), e um 1921 em `ActualYear` não fica isolado, porque o hiato até ao ano seguinte é inferior a 42 anos.

#### 4.4.6 Errors and outliers are different things. From this dataset, give one example of each: an error that is also an outlier; an error that no univariate method flags and a rule over two columns does; an error that no rule finds and only the joint distribution reveals; an outlier that is not an error.

In [ ]:
publicada = pd.read_csv(Path("build") / "MERGE_ECAC" / "merge_original" / "metadata.csv").drop_duplicates("Audio_Song").set_index("Audio_Song")
def publicado(i, col):
    return publicada.loc[i, col] if i in publicada.index else np.nan

# erro que também é outlier: valor isolado, marcado pelos três critérios
errosIsolados = isolados(treino_c["Relevance"], caudas["Relevance"])
e1 = errosIsolados[errosIsolados].index[0]

# erro que nenhum método univariado marca e uma regra sobre duas colunas apanha: ActualYear < LowestYear
ay_bruto = pd.to_numeric(sem_dup.set_index("Audio_Song").loc[treino_c.index, "ActualYear"], errors="coerce")
ly = treino_c["LowestYear_y"]
viola_par = (ay_bruto < ly) & ~ay_bruto.isin([0, 1900])
z, modz, tukey, _ = criterios(ay_bruto, treino_c["ActualYear"])
sem_marca = viola_par & ~z & ~modz & ~tukey
print(f"ActualYear < LowestYear: {int(viola_par.sum())} registos; dos quais {int(sem_marca.sum())} sem marca em nenhum dos três critérios univariados")
e2 = sem_marca[sem_marca].index[0]

# erro que nenhuma regra apanha e só a distribuição conjunta revela: os melhores candidatos de 4.4.4 contra a tabela publicada
cols_num = ["Relevance", "Duration", "ActualYear", "Year", "LowestYear"]
conf = []
for i in top.index:
    if i in publicada.index:
        dif = {c: (publicado(i, c), sem_dup.set_index("Audio_Song").loc[i, c]) for c in cols_num
                if str(publicado(i, c)) != str(sem_dup.set_index("Audio_Song").loc[i, c]) and not (pd.isna(publicado(i, c)) and pd.isna(sem_dup.set_index("Audio_Song").loc[i, c]))}
        conf.append({"registo": i, "título": treino_c.loc[i, "Title"], "colunas diferentes da publicada (publicado, atual)": dif or "nenhuma"})
display(pd.DataFrame(conf).set_index("registo"))
e3 = "MT0033459750"

# outlier que não é erro: o grupo de Relevance aproximadamente 1.5 e a música mais longa que sobrevive ao critério
grupo_ids = treino_c.index[(treino_c["Relevance"] > 1.4) & (treino_c["Relevance"] < 1.7)]
iguais = sum(np.isclose(publicado(i, "Relevance"), treino_c.loc[i, "Relevance"]) for i in grupo_ids)
e4 = treino_c["Duration"].idxmax()
print(f"grupo Relevance aproximadamente 1.5: {len(grupo_ids)} músicas, {iguais} com o valor igual ao publicado | música mais longa que sobrevive: {e4} ({treino_c.loc[e4, 'Duration']:.0f} s, publicado: {publicado(e4, 'Duration'):.0f} s)")

exemplos = pd.DataFrame([
    ["erro e outlier", e1, "Relevance", treino_c.loc[e1, "Relevance"], publicado(e1, "Relevance"), "marcado pelos 3 critérios; cauda isolada"],
    ["erro sem marca univariada, apanhado por regra de duas colunas", e2, "ActualYear / LowestYear", f"{ay_bruto[e2]:.0f} / {ly[e2]:.0f}", f"{publicado(e2, 'ActualYear')} / {publicado(e2, 'LowestYear')}", "ActualYear < LowestYear"],
    ["erro sem regra, visível só na distribuição conjunta", e3, "LowestYear (com Year e Duration)", f"{treino_c.loc[e3, 'LowestYear_y']:.0f}", publicado(e3, "LowestYear"), "isolado no varrimento por pares"],
    ["outlier que não é erro", e4, "Duration", f"{treino_c.loc[e4, 'Duration']:.0f}", f"{publicado(e4, 'Duration'):.0f}", "extremo, igual à tabela publicada"],
], columns=["exemplo", "registo", "coluna", "valor na tabela", "valor publicado", "como se vê"]).set_index("exemplo")
display(exemplos)

fig, (a1, a2, a3, a4) = plt.subplots(1, 4, figsize=(18, 4))
r = treino_c["Relevance"].dropna()
a1.hist(r[r < 1.4], bins=30, color="grey")
a1.hist(r[(r > 1.4) & (r < 1.7)], bins=5, color="tab:blue", label="grupo (32)")
a1.axvline(treino_c.loc[e1, "Relevance"], color="red", label="erro e outlier")
a1.set_title("1) Relevance")
a1.legend(fontsize=8)
a1.set_yscale("log")

a2.scatter(ly, ay_bruto.where(ay_bruto > 1900), s=8, color="grey", alpha=0.4)
a2.scatter(ly[sem_marca], ay_bruto[sem_marca], s=22, color="red", label="ActualYear < LowestYear")
a2.plot([1950, 2010], [1950, 2010], "k--", lw=0.7)
a2.set_xlabel("LowestYear")
a2.set_ylabel("ActualYear")
a2.set_title("2) regra de duas colunas")
a2.legend(fontsize=8)

Zp = treino_c[["Duration", "LowestYear_y"]].dropna()
a3.scatter(Zp["LowestYear_y"], Zp["Duration"], s=8, color="grey", alpha=0.4)
a3.scatter(treino_c.loc[e3, "LowestYear_y"], treino_c.loc[e3, "Duration"], s=90, color="red")
a3.set_xlabel("LowestYear")
a3.set_ylabel("Duration (s)")
a3.set_title("3) só a distribuição conjunta")

d = treino_c["Duration"].dropna()
a4.boxplot(d, vert=False, whis=1.5, showfliers=False); a4.scatter(treino_c.loc[e4, "Duration"], 1, color="tab:blue", s=70, label="extremo correto"); a4.set_yticks([])
a4.set_title("4) outlier que não é erro")
a4.legend(fontsize=8)
plt.tight_layout()
plt.show()

Os quatro exemplos, com o valor publicado em `merge_original` como confirmação (a avaliação sistemática dos detetores é em 6):

| Exemplo | Registo | O que se vê | Confirmação |
|:--|:--|:--|:--|
| erro que também é outlier | `MT0004184616` | `Relevance` = 1,84, marcada pelos três critérios, cauda isolada. Nenhuma regra de 4.3 a apanha porque é positiva e finita | o valor publicado é 1,041 |
| erro que nenhum método univariado marca e uma regra de duas colunas apanha | `MT0009321428` | `ActualYear` 1976 com `LowestYear` 1996. Dos 32 registos de treino com `ActualYear` < `LowestYear`, 28 não têm marca em nenhum dos três critérios | o valor publicado é 1996 |
| erro que nenhuma regra apanha e só a distribuição conjunta mostra | `MT0033459750` | `LowestYear` 1962 com `Duration` 345 s e `Year` 1967: banal em cada coluna, isolado no par `Duration` × `LowestYear` | o valor publicado é 1967-12-?? |
| outlier que não é erro | `MT0001435061`, e o grupo de `Relevance` ≈ 1,5 | `Duration` de 656 s (a maior do treino) e 32 músicas de `Relevance` de 1,48 a 1,50 | todos iguais à tabela publicada |

O que os separa é a pergunta que cada um responde. O erro viola uma regra, e pode ser invisível se for um valor ordinário numa coluna e impossível ao lado de outra (segundo exemplo). O outlier só está longe dos outros e pode ser verdadeiro (quarto exemplo). Os dois conjuntos cruzam-se (primeiro exemplo) e deixam áreas que um só método não cobre: o terceiro exemplo foi um dos quatro candidatos mais isolados do varrimento e, dos quatro, o único com um valor alterado que escapava às regras. Por isso um filtro de outliers não substitui as regras, e as regras não substituem a distribuição conjunta.

### 4.5 Media

#### 4.5.1 Inspect the covers and the audio. Repeated images, missing covers, excerpts that are not what they claim. Does any of it threaten the evaluation?

In [ ]:
from PIL import Image
import mutagen
import librosa

assert (MEDIA / "album_covers").exists() and (MEDIA / "audio").exists(), "coloque a pasta de média em MEDIA"

ids_tabela = set(corrigido["Audio_Song"])
ids_teste = set(teste["Audio_Song"])
quadrante = meta.drop_duplicates("Audio_Song").set_index("Audio_Song")["Quadrant"]

# capa
def hashPerceptual(img, n=8):
    p = np.asarray(img.convert("L").resize((n, n), Image.LANCZOS), dtype=float)
    return "".join("1" if v else "0" for v in (p > p.mean()).ravel())

linhas = []
for p in (MEDIA / "album_covers").rglob("*.jpg"):
    with Image.open(p) as im:
        w, h = im.size
        g = np.asarray(im.convert("L"), dtype=float)
        linhas.append({"id": p.stem, "largura": w, "altura": h, "tamanho_kb": p.stat().st_size / 1024,
                       "md5": hashlib.md5(p.read_bytes()).hexdigest(), "ahash": hashPerceptual(im), "desvio_cinzentos": g.std()})
capas = pd.DataFrame(linhas).set_index("id")
capas["na_tabela"] = capas.index.isin(ids_tabela)
capas["teste"] = capas.index.isin(ids_teste)

print(f"capas: {len(capas)} | de músicas da tabela: {int(capas.na_tabela.sum())} | músicas da tabela sem capa: {len(ids_tabela - set(capas.index))}")
sem_capa = pd.Series(sorted(ids_tabela - set(capas.index)), name="id").to_frame()
sem_capa["quadrante"] = sem_capa["id"].map(quadrante)
sem_capa["teste"] = sem_capa["id"].isin(ids_teste)
display(pd.crosstab(sem_capa["quadrante"], sem_capa["teste"], margins=True))

uteis = capas[capas.na_tabela]
for chave in ("md5", "ahash"):
    g = uteis.groupby(chave)
    rep = g.filter(lambda x: len(x) > 1)
    atravessam = g.filter(lambda x: len(x) > 1 and x["teste"].any() and (~x["teste"]).any())
    print(f"{chave}: {rep[chave].nunique()} imagens repetidas ({len(rep)} músicas) | grupos com músicas dos dois lados do teste: {atravessam[chave].nunique()} ({int(atravessam['teste'].sum())} músicas de teste)")
print("dimensões mais frequentes:", capas[["largura", "altura"]].value_counts().head(5).to_dict())
print("capas quase uniformes (desvio de cinzentos < 5):", int((capas["desvio_cinzentos"] < 5).sum()))

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(capas["largura"], capas["altura"], s=8, alpha=0.4)
m = max(capas["largura"].max(), capas["altura"].max())
ax.plot([0, m], [0, m], "k--", lw=0.7)
ax.set_xlabel("largura (px)")
ax.set_ylabel("altura (px)"); ax.set_title("capas: largura × altura")
plt.tight_layout()
plt.show()

In [ ]:
# áudio
linhas = []
for p in (MEDIA / "audio").rglob("*.mp3"):
    info = mutagen.File(p).info
    row = {"id": p.stem, "duracao_s": info.length, "taxa_amostragem": info.sample_rate, "canais": info.channels,
           "bitrate_kbps": info.bitrate / 1000, "tamanho_kb": p.stat().st_size / 1024, "md5": hashlib.md5(p.read_bytes()).hexdigest()}
    if p.stem in ids_tabela:
        y, sr = librosa.load(p, sr=22050, mono=True)
        rootMeanSq = librosa.feature.rms(y=y)[0]
        row.update({"rms_medio": float(rootMeanSq.mean()), "pico": float(np.abs(y).max()),
                    "fracao_silencio": float((rootMeanSq < 1e-3).mean()), "fracao_saturada": float((np.abs(y) > 0.99).mean())})
    linhas.append(row)
audio = pd.DataFrame(linhas).set_index("id")
audio["na_tabela"] = audio.index.isin(ids_tabela)
stats = ["duracao_s", "taxa_amostragem", "canais", "bitrate_kbps", "rms_medio", "pico", "fracao_silencio", "fracao_saturada"]
display(audio[stats].describe().T)

fig, eixos = plt.subplots(2, 4, figsize=(16, 6.5))
for ax, col in zip(eixos.ravel(), stats):
    audio[col].dropna().plot.hist(bins=40, ax=ax); ax.set_title(col)
plt.tight_layout()
plt.show()
print("excertos que não têm ~30 s (fora de 29–31 s):", int((~audio["duracao_s"].between(29, 31)).sum()),
      "| silêncio > 50 %:", int((audio["fracao_silencio"] > 0.5).sum()))

Capas: há 2019 capas, 1211 de músicas da tabela e 108 músicas da tabela sem capa. Os números diferem um pouco de 2.2 (109 e 805), o que deve vir de a tabela já não ter as cópias exatas nem as 4 linhas retiradas em 4.1.

- Capas em falta: faltam 88 músicas de treino (9,7 %) e 20 de teste (4,9 %). No treino a falta é parecida em todos os quadrantes (Q1 9,1 %, Q2 7,2 %, Q3 11,0 %, Q4 10,1 %), por isso não parece carregar informação sobre o alvo. No teste as taxas variam mais (Q1 1,9 %, Q2 10,7 %, Q3 5,9 %, Q4 4,9 %), mas com 3 a 9 casos por quadrante não se tira conclusão. O que fica é que o teste tem cerca de metade da falta do treino.
- Capas repetidas: Há 83 imagens repetidas pelo `md5` (189 músicas) e 87 pelo hash perceptual (198 músicas). Atravessam a fronteira do teste 30 grupos pelo `md5` (34 músicas de teste) e 34 pelo hash perceptual (39 músicas de teste), ou seja, cerca de 9 % das músicas de teste têm no treino uma música com a mesma capa. Isto ameaça a avaliação: uma capa partilhada costuma ser o mesmo álbum ou artista, e um modelo com atributos de imagem pode reconhecê-la em vez de generalizar, tal como acontecia com as letras iguais de 4.1.4. Ainda não se sabe se essas músicas partilham a etiqueta.
- Dimensões e imagens por defeito: 1386 capas têm 600×600 e 71 têm 1200×1200, mas há muitas quase quadradas (600×595, 600×596, 600×597...), por isso as capas têm de se redimensionar antes de extrair atributos. Só uma capa é quase uniforme (desvio de cinzentos < 5), não tendo peso na análise.

Áudio: há 2216 ficheiros, 1319 com linha na tabela (os que têm medidas de energia).

- Duração: a mediana é 30,06 s, mas 39 excertos ficam fora de 29–31 s e o mais curto tem 4,8 s. Excertos truncados dão atributos como o ritmo calculados com muito menos áudio. Não se sabe ainda quantos dos 39 pertencem à tabela. A marca `audio_fora_30s` de 4.6 conta-os.
- Formato: a taxa de amostragem vai de 22 050 a 44 100 Hz (pelo menos metade a 32 000), o débito de 64 a 128 kbps (pelo menos metade a 96) e 2 ficheiros são mono. Estas diferenças mudam o conteúdo em frequências altas, por isso é preciso reamostrar para uma taxa comum antes de extrair atributos. Se a taxa ou o débito dependerem do quadrante ou da partição, isso é uma fuga de codificação. Isso ainda não foi verificado.
- Energia e saturação: o root-mean-squared varia de 0,014 a 0,39. A energia é informação útil para a ativação (arousal), por isso não se normaliza o volume sem pensar. O pico passa de 1,0 em pelo menos 25 % dos ficheiros (máximo 1,39), o que é típico do descodificador de mp3 e não saturação: a fração de amostras acima de 0,99 é quase nula (máximo 1,2 %). Nenhum excerto está vazio (silêncio máximo de 5,9 %, nenhum acima de 50 %).
- Não se verificou se cada excerto corresponde à música indicada, porque isso exigia ouvi-los.

Ameaça à avaliação: a principal são as capas repetidas entre treino e teste. As capas em falta e as diferenças de formato do áudio são problemas de pré-processamento, a confirmar com a verificação de fuga por codificação.

Decisões que entram em 4.6: não se apaga nenhuma música. Acrescentam-se `sem_capa` (a capa não se imputa), `capa_grupo` e `capa_repetida` (a chave de grupo é o hash perceptual, usada na validação junto com `Artist_norm`) e `audio_fora_30s` (decide-se em 10 se esses excertos se descartam).

### 4.6 The cleaned dataset

#### 4.6.1 Apply the decisions of 4.1 to 4.5 and produce the cleaned dataset. The test partition is cleaned by the same rules, fitted on the rest.

In [ ]:
SAIDA = Path("build") / "MERGE_ECAC" / "cleaned"
SAIDA.mkdir(parents=True, exist_ok=True)

def registar_outliers(d, nome):
    """Valores isolados -> NaN (4.4.5). Marcas informativas do DBSCAN e do k-means, sem apagar linhas."""
    d = d.copy()
    log = {}
    for c in COLS_MEDIDA:
        m = isolados(d[c], caudas[c])
        log[f"{NOMES.get(c, c)}: valor isolado -> NaN"] = int(m.sum())
        d.loc[m, c] = np.nan
    sub = d[COLS_MV].dropna()
    X = escala.transform(sub)
    idx = d[COLS_MV].dropna().index
    # DBSCAN aplicado a pontos novos: ruído se nenhum ponto nuclear do treino estiver a menos de eps
    dist_nucleo = NearestNeighbors(n_neighbors=1).fit(db.components_).kneighbors(X)[0][:, 0]
    d["isolado_dbscan"] = False
    d.loc[idx, "isolado_dbscan"] = dist_nucleo > EPS
    dist_km = np.linalg.norm(X - km.cluster_centers_[km.predict(X)], axis=1)
    d["longe_kmeans"] = False
    d.loc[idx, "longe_kmeans"] = dist_km > LIMITE_KM
    d["n_criterios_uni"] = 0
    return d, log

# a mesma pipeline aplicada separadamente a cada partição, para contar o custo em cada uma
e_teste_bruta = sem_dup["Audio_Song"].isin(teste["Audio_Song"])
partes = {}
logs_43 = {}
logs_44 = {}
for nome, parte in (("treino", sem_dup[~e_teste_bruta]), ("teste", sem_dup[e_teste_bruta])):
    col, logs_43[nome] = corrigir_valores(parte)
    col = acrescentar_derivadas(col).set_index("Audio_Song")
    partes[nome], logs_44[nome] = registar_outliers(col, nome)
    assert not (partes[nome]["Duration"] <= 0).any() and not (partes[nome]["ActualYear"] > 2026).any()

# contagens das marcas univariadas (três critérios) também no teste, com parâmetros do treino
for nome, d in partes.items():
    n = pd.Series(0, index=d.index)
    for col in COLS_UNI:
        z, modz, tukey, _ = criterios(d[col], treino_c[col])
        n = n + (z | modz | tukey).astype(int)
    d["n_criterios_uni"] = n.astype(int)

limpo = pd.concat([partes["treino"].assign(particao="treino"), partes["teste"].assign(particao="teste")])
# 4.2.2: colunas quase vazias saem; as colunas auxiliares de contagem não seguem (derivam das etiquetas)
a_sair = ["num_Genres", "num_MoodsAll", "n_Moods", "n_MoodsAll", "n_Appearances"]
limpo = limpo.drop(columns=a_sair).reset_index()
for nome in ("treino", "teste"):
    limpo[limpo["particao"] == nome].drop(columns="particao").to_csv(SAIDA / f"metadata_clean_{nome}.csv", index=False)
limpo.to_csv(SAIDA / "metadata_clean.csv", index=False)

# verificações
assert limpo["Audio_Song"].is_unique
assert set(limpo.loc[limpo.particao == "teste", "Audio_Song"]) == set(teste["Audio_Song"]), "o teste tem de ficar com as mesmas 409 músicas"
assert not set(REMOVER_FRONTEIRA) & set(limpo["Audio_Song"])
print(f"dataset limpo: {len(limpo)} linhas x {limpo.shape[1]} colunas | treino {int((limpo.particao == 'treino').sum())} | teste {int((limpo.particao == 'teste').sum())}")
print("guardado em", SAIDA)

O dataset limpo tem 1319 linhas, 910 de treino e 409 de teste, e 32 colunas. As 409 músicas do teste são as mesmas da partição original (a célula verifica-o): nada se removeu do teste a não ser as 7 cópias exatas, que eram linhas repetidas de músicas que continuam lá. Ficam as colunas originais (menos as duas quase vazias), mais `Artist_norm` (para agrupar artistas na validação), `Year_y` e `LowestYear_y` (o ano de cada data), e as marcas `isolado_dbscan`, `longe_kmeans` e `n_criterios_uni`, que informam mas não apagam. Ficam também em ficheiros separados, `metadata_clean_train.csv` e `metadata_clean_test.csv`.

#### 4.6.2 What did each decision cost? Rows removed, values replaced, information lost.

In [ ]:
antes = {n: p for n, p in (("treino", sem_dup[~e_teste_bruta]), ("teste", sem_dup[e_teste_bruta]))}
def _buscarLog(nome, chave):
    return {**logs_43[nome], **logs_44[nome]}.get(chave, 0)
def buscarLog(chave):
    return _buscarLog("treino", chave), _buscarLog("teste", chave)

cop = (log_41["4.1 cópias exatas (treino)"], log_41["4.1 cópias exatas (teste)"])
linhas = [
 ("4.1", "Remover cópias exatas", "linhas removidas", *cop, "nenhuma: eram o mesmo registo"),
 ("4.1", "Remover do treino as músicas que têm par no teste (mesma gravação ou letra)", "linhas removidas", 3, 0, "3 exemplos rotulados (0,3 % do treino)"),
 ("4.1", "Mesma gravação no mesmo lado: ficar com a linha mais completa", "linhas removidas", 1, 0, "1 exemplo; a linha removida era a mais vazia"),
 ("4.1", "Artist_norm: coluna nova para agrupar artistas", "colunas acrescentadas", 1, 1, "nenhuma (Artist mantém-se)"),
 ("4.3", "Duration: sentinela 9999 e valores não positivos -> NaN", "valores que passam a faltar", buscarLog("Duration: sentinela 9999 -> NaN")[0] + buscarLog("Duration: zero ou negativa -> NaN")[0],
        buscarLog("Duration: sentinela 9999 -> NaN")[1] + buscarLog("Duration: zero ou negativa -> NaN")[1], "informação perdida: voltam a ser faltas (imputar dentro da validação)"),
 ("4.3", "Duration: milissegundos -> segundos, minutos -> segundos", "valores substituídos", buscarLog("Duration: milissegundos -> segundos")[0] + buscarLog("Duration: minutos -> segundos")[0],
        buscarLog("Duration: milissegundos -> segundos")[1] + buscarLog("Duration: minutos -> segundos")[1], "nenhuma se a conversão estiver certa; no caso dos minutos o valor é arredondado ao segundo"),
 ("4.3", "ActualYear: malformado, sentinelas 0/1900/9999 e futuro -> NaN", "valores que passam a faltar",
        buscarLog("ActualYear: malformado -> NaN")[0] + buscarLog("ActualYear: sentinela 0/1900/9999 -> NaN")[0] + buscarLog("ActualYear: no futuro -> NaN")[0],
        buscarLog("ActualYear: malformado -> NaN")[1] + buscarLog("ActualYear: sentinela 0/1900/9999 -> NaN")[1] + buscarLog("ActualYear: no futuro -> NaN")[1], "informação perdida"),
 ("4.3", "ActualYear anterior a LowestYear -> NaN (Year concorda com LowestYear)", "valores que passam a faltar", *buscarLog("ActualYear: anterior a LowestYear -> NaN"), "informação perdida; o ano certo podia ser recuperado de Year"),
 ("4.3", "Pesos com tamanho diferente da lista de etiquetas -> NaN", "valores que passam a faltar",
        sum(v for k, v in {**logs_43["treino"]}.items() if "Weights" in k), sum(v for k, v in {**logs_43["teste"]}.items() if "Weights" in k), "informação perdida: não se sabe que peso é de que etiqueta"),
 ("4.4", "Valores isolados e sem grupo -> NaN (Relevance, Duration, ActualYear, Year, LowestYear)", "valores que passam a faltar",
        sum(v for k, v in logs_44["treino"].items() if "isolado" in k), sum(v for k, v in logs_44["teste"].items() if "isolado" in k), "mínima; os extremos corretos e o grupo de Relevance ≈ 1,5 ficam"),
 ("4.4", "Marcas isolado_dbscan, longe_kmeans, n_criterios_uni (nada se apaga)", "colunas acrescentadas", 3, 3, "nenhuma"),
 ("4.2", "Remover num_Genres e num_MoodsAll (99 % em falta)", "colunas removidas", 2, 2, "nenhuma útil: 10 valores em 1 323"),
]
custo = pd.DataFrame(linhas, columns=["secção", "decisão", "unidade", "treino", "teste", "custo em informação"]).set_index(["secção", "decisão"])
display(custo)

taxa = pd.DataFrame({
    "em falta antes (%)": {c: round(antes["treino"][c].isna().mean() * 100, 1) for c in ["Duration", "ActualYear", "Relevance", "GenreWeights", "MoodsAllWeights", "StyleWeights", "ThemeWeights"]},
    "em falta depois (%)": {c: round(partes["treino"][c].isna().mean() * 100, 1) for c in ["Duration", "ActualYear", "Relevance", "GenreWeights", "MoodsAllWeights", "StyleWeights", "ThemeWeights"]}})
display(taxa.assign(**{"antes, após remover duplicados (%)": [round(sem_dup[~e_teste_bruta][c].isna().mean() * 100, 1) for c in taxa.index]})[["antes, após remover duplicados (%)", "em falta depois (%)"]])

A tabela anterior tem uma linha por decisão. Em resumo:
- Linhas: saem 25 cópias exatas (18 do treino e 7 do teste, sem perda de informação), 3 músicas de treino que têm a mesma gravação ou a mesma letra no teste, e 1 linha do treino que repetia a gravação de outra. Perdem-se 4 exemplos rotulados do treino (0,4 %) e nenhum do teste.
- Valores que passam a faltar: 120 durações (sentinela 9999 ou zero/negativas), 50 `ActualYear` (malformado, sentinelas e futuro) e 40 `ActualYear` anteriores a `LowestYear`, 45 listas de pesos que não batem com as etiquetas, e 2 valores isolados (um `Relevance` no treino, uma `Duration` no teste). A informação que se perde aqui é a mais cara: no treino a taxa de falta de `Duration` sobe de 20,2 % para 29,9 % e a de `ActualYear` de 23,1 % para 30,3 %.
- Valores substituídos: 85 durações recuperadas (60 em milissegundos divididas por 1000, 25 em minutos multiplicadas por 60 e arredondadas ao segundo), em vez de as deitar fora. O risco é o arredondamento e uma conversão errada se uma música tiver mesmo menos de 30 s.
- Colunas: saem 2 (`num_Genres`, `num_MoodsAll`, com 99 % em falta) e entram 6 (`Artist_norm`, `Year_y`, `LowestYear_y` e as 3 marcas), mais a coluna `particao`.

Há ainda um custo que não aparece na tabela: o `ActualYear` anterior a `LowestYear` podia ser recuperado de `Year` (concordam em 53 de 55 casos), mas optou-se por NaN para não inventar valores antes de 6, onde se mede o que se perdeu e o que se acertou.

## 5. Class distribution
### 5.1 Report the distribution and the ratio between the largest and smallest class

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

print(u.columns.tolist())

def distribuicao(serie, nome):

    c = serie.value_counts().sort_index()  # Conta as músicas de cada quadrante e ordenar os resultados

    print(f"{nome}: {c.sum()} músicas")  # Mostra o número total de músicas

    # Cria uma tabela com o número de músicas e a respetiva percentagem por quadrante
    print(pd.DataFrame({
        "n": c,
        "%": (c / c.sum() * 100).round(1)
    }))

    # Calcula e apresenta o rácio entre o quadrante com mais músicas e o com menos músicas
    print(f"rácio maior/menor: {c.max() / c.min():.2f} : 1\n")

    return c  #Devolve as contagens para poderem ser utilizadas posteriormente

# Seleciona apenas as músicas que pertencem ao conjunto de treino, excluindo as de teste
y = u.loc[~u["teste"], "Quadrant"]

print("treino:", len(y))
antes = distribuicao(meta["Quadrant"], f"Como veio ({len(meta)})")
sem_exatos = distribuicao(u["Quadrant"], f"Sem duplicados exatos ({len(u)})")
contagem_treino = distribuicao(y, f"Treino ({len(y)})")

pd.DataFrame({
    "como veio": antes,
    "sem duplicados exatos": sem_exatos
}).plot.bar(rot=0)

plt.ylabel("músicas")
plt.title("Distribuição por quadrante")
plt.tight_layout()
plt.show()

A distribuição das classes apresenta algum desequilíbrio. A classe Q1 é a mais representada, seguida da Q3, enquanto Q4 é a menos representada.

Após a remoção dos 25 registos duplicados, o dataset passou de 1348 para 1323 músicas. O rácio entre a classe maioritária (Q1) e a minoritária (Q4) diminuiu ligeiramente de 4,09:1 para 4,04:1.

Conclui-se que existe um desequilíbrio moderado entre as quatro classes, que deverá ser considerado na avaliação dos modelos, nomeadamente através da métrica Macro F1.


### 5.2 Which metrics stop being informative at that ratio?

In [ ]:
maioria = y.value_counts().idxmax()
print("classe maioritária:", maioria)
print("acertar só as duas maiores classes:", round(y.value_counts().nlargest(2).sum() / len(y), 3))

A métrica Accuracy, quando analisada isoladamente, pode ser enganadora devido ao desequilíbrio entre classes. Neste caso, a classe maioritária é Q1 e representa uma parte significativa dos dados.

Além disso, as duas classes mais frequentes (Q1 e Q3) representam 76,3% das músicas, o que significa que um modelo pode obter uma accuracy aparentemente elevada sem classificar corretamente as classes minoritárias.

Por isso, é importante utilizar também métricas como Macro F1, Precision e Recall, que permitem avaliar melhor o desempenho do modelo em todas as classes.


## 5.3 What does a majority-class classifier obtain?

In [ ]:
# Calcula a accuracy do classificador que prevê sempre a classe maioritária
acc = (y == maioria).mean()

# Calcula o Macro F1 teórico do classificador de classe maioritária
f1_macro = (2 * acc / (acc + 1)) / y.nunique()

print("fórmula | accuracy:", round(acc, 3), "| macro F1:", round(f1_macro, 3))

r = cross_validate(
    DummyClassifier(strategy="most_frequent"),  # Cria um classificador que prevê sempre a classe mais frequente
    np.zeros((len(y), 1)),  # Cria uma matriz fictícia de dados, pois o classificador não utiliza características
    y,  # Define Quadrant como variável que queremos prever
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),  # Dividie os dados em 5 grupos, mantendo a proporção das classes
    scoring=["accuracy", "f1_macro"]  # Avaliar o modelo através da Accuracy e do Macro F1
)

print("5 folds | accuracy:", round(r["test_accuracy"].mean(), 3), "| macro F1:", round(r["test_f1_macro"].mean(), 3))

Um classificador que prevê sempre a classe maioritária (Q1) obtém uma Accuracy de 40% e um Macro F1 de 0,14.

Os resultados foram confirmados através de validação cruzada com 5 folds, apresentando os mesmos valores.

Conclui-se que um modelo que prevê apenas a classe mais frequente tem um desempenho bastante limitado, especialmente na classificação equilibrada das quatro classes.